# LEP-20 — Préparer les données pour la modélisation prédictive de l’espérance de vie

**Projet** : WHO Life Expectancy  
**Sprint** : 4  
**Période des données** : 2000–2015  
**Source** : `data/processed/life_expectancy_fiabilise_sanitaire_socioeducatif.csv`

## Objectif

Définir le scénario de prédiction, sélectionner les variables disponibles
au moment de la prédiction et préparer les ensembles d’entraînement et
de test qui serviront à comparer les modèles dans les stories suivantes.

### Hypotheses de travail


Pour chaque pays et chaque année, nous estimons l’espérance de vie à la
naissance à partir des autres indicateurs de la même année. L’analyse
est rétrospective. Par exemple, pour estimer l’espérance de vie en 2015,
nous supposons que les indicateurs retenus de 2015 sont déjà disponibles.
La valeur observée de l’espérance de vie est conservée pour mesurer l’erreur.

| Période | Utilisation |
|---|---|
| 2000–2011 | Entraîner les modèles candidats. |
| 2012–2013 | Comparer les modèles et choisir leurs réglages. |
| 2000–2013 | Réentraîner le modèle retenu avec les réglages choisis. |
| 2014–2015 | Évaluer le modèle final sur deux années réservées. |


`life_expectancy` est la **variable cible**. Elle est placée dans `y` et
écartée des entrées `X`. Parmi les autres colonnes du jeu final, aucune
n’a été identifiée comme étant calculée directement à partir de cette
valeur. Le fichier contient le RNB par habitant, une mesure économique
distincte de l’IDH complet, qui intègre l’espérance de vie.

Les taux de mortalité restent des variables candidates. 

Nous examinons les types et les valeurs manquantes sur la période
d’entraînement de 2000 à 2011. 

## 1. Chargement du jeu de données et selection des variables

In [2]:
# Bibliothèques
from zipfile import ZipFile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import pandas as pd

# Charger le jeu fiabilisé et définir la colonne cible.
df = pd.read_csv(
    "../data/processed/life_expectancy_fiabilise_sanitaire_socioeducatif.csv"
)
variable_cible = "life_expectancy"

# Limiter l'audit aux années d'entraînement des modèles candidats.
df_entrainement = df.loc[df["year"].between(2000, 2011)]

# Résumer le type et la couverture de chaque colonne.
audit_variables = pd.DataFrame({
    "type": df_entrainement.dtypes.astype(str),
    "valeurs_manquantes": df_entrainement.isna().sum(),
    "pct_manquant": df_entrainement.isna().mean().mul(100).round(2),
})

print(f"Observations dans la période d'entraînement : {len(df_entrainement)}")
display(audit_variables.sort_values("pct_manquant", ascending=False))

Observations dans la période d'entraînement : 2196


,type,valeurs_manquantes,pct_manquant
hepatitis_b,float64,516,23.50
measles_reported_incidence_per_million,float64,112,5.10
health_expenditure_pc_usd,float64,77,3.51
expected_years_of_schooling,float64,64,2.91
total_expenditure,float64,39,1.78
gni_per_capita_ppp_2021,float64,28,1.28
thinness_5_9_years,float64,24,1.09
thinness_10_19_years,float64,24,1.09
gdp,float64,20,0.91
diphtheria,float64,19,0.87


## 2. Sélection des variables d’entrée

Nous retenons 18 variables candidates, soit 17 indicateurs numériques
et une variable catégorielle (`status`). Cette dernière représente
la catégorie de développement telle qu’elle figure dans le jeu de
données. Elle sera encodée dans le pipeline de modélisation.

Nous écartons `total_expenditure` parce que sa définition reste incertaine.
Les dépenses de santé sont représentées par `health_expenditure_pc_usd`,
un indicateur documenté de dépenses par habitant.

`country` et `iso3` servent à identifier les pays, `year` sert au découpage
temporel, et `life_expectancy` est la variable cible.

In [4]:
# Séparer la cible, les identifiants et la colonne écartée.
colonnes_identification = ["country", "iso3", "year"]
colonnes_ecartees = ["total_expenditure"]

colonnes_hors_modele = (
    [variable_cible] + colonnes_identification + colonnes_ecartees
)
variables_entree = df.columns.drop(colonnes_hors_modele).to_list()

# Distinguer les entrées pour préparer le prétraitement.
variables_categorielles = ["status"]
variables_numeriques = [
    colonne for colonne in variables_entree
    if colonne not in variables_categorielles
]

print(f"Variables d'entrée candidates : {len(variables_entree)}")
display(audit_variables.loc[variables_entree].sort_values(
    "pct_manquant", ascending=False
))

# Vérifier les catégories présentes dans l'entraînement.
display(df_entrainement["status"].value_counts(dropna=False))

Variables d'entrée candidates : 18


,type,valeurs_manquantes,pct_manquant
hepatitis_b,float64,516,23.50
measles_reported_incidence_per_million,float64,112,5.10
health_expenditure_pc_usd,float64,77,3.51
expected_years_of_schooling,float64,64,2.91
gni_per_capita_ppp_2021,float64,28,1.28
thinness_10_19_years,float64,24,1.09
thinness_5_9_years,float64,24,1.09
gdp,float64,20,0.91
polio,float64,19,0.87
diphtheria,float64,19,0.87


status
Developing    1812
Developed      384
Name: count, dtype: int64

### 3. Découpage temporel des données

Nous entraînons les modèles candidats sur 2000–2011, comparons leurs
résultats sur 2012–2013 et réservons 2014–2015 au test final.

Nous gardons les observations dont l’espérance de vie est renseignée.
Les valeurs manquantes des variables d’entrée restent dans `X`.
L’imputation et l’encodage de `status` seront ajustés sur l’entraînement
dans le pipeline.

In [5]:
# Garder les observations pour lesquelles la cible est connue.
df_modelisable = df.dropna(subset=[variable_cible])

# Répartir les observations selon leur année.
df_entrainement = df_modelisable.loc[df_modelisable["year"].between(2000, 2011)]
df_validation = df_modelisable.loc[df_modelisable["year"].between(2012, 2013)]
df_test = df_modelisable.loc[df_modelisable["year"].between(2014, 2015)]

# Séparer les 18 variables d'entrée de la cible dans chaque ensemble.
X_train = df_entrainement[variables_entree]
y_train = df_entrainement[variable_cible]

X_val = df_validation[variables_entree]
y_val = df_validation[variable_cible]

X_test = df_test[variables_entree]
y_test = df_test[variable_cible]

print(f"Observations écartées faute de cible : {len(df) - len(df_modelisable)}")
print(f"Entraînement : {X_train.shape}")
print(f"Validation   : {X_val.shape}")
print(f"Test         : {X_test.shape}")

Observations écartées faute de cible : 2
Entraînement : (2196, 18)
Validation   : (374, 18)
Test         : (366, 18)


In [6]:
from pathlib import Path

# Conserver les identifiants, les entrées et la cible.
colonnes_export = colonnes_identification + variables_entree + [variable_cible]

df_modelisation = pd.concat([
    df_entrainement[colonnes_export].assign(ensemble="entrainement"),
    df_validation[colonnes_export].assign(ensemble="validation"),
    df_test[colonnes_export].assign(ensemble="test"),
], ignore_index=True)

# Enregistrer le découpage pour les prochains notebooks.
fichier_modelisation = Path("../data/processed/life_expectancy_modelisation.csv")
df_modelisation.to_csv(fichier_modelisation, index=False, encoding="utf-8-sig")

print(f"Fichier exporté : {fichier_modelisation}")
print(f"Dimensions : {df_modelisation.shape}")

Fichier exporté : ..\data\processed\life_expectancy_modelisation.csv
Dimensions : (2936, 23)


### 4. Hypothèse de traitement des valeurs manquantes

Hypothèse de traitement des valeurs manquantes

Nous faisons l’hypothèse que, pour un indicateur donné, les valeurs
observées chez les pays de même statut de développement (`Developed`
ou `Developing`) fournissent un meilleur point de référence que celles
de l’ensemble des pays. Par exemple, si le PIB par habitant est manquant
pour un pays classé `Developing`, nous utilisons la médiane du PIB par
habitant observé dans cette catégorie.

Les médianes sont calculées uniquement sur les données d’entraînement
de 2000 à 2011, puis appliquées à l’entraînement et à la validation.
Si le statut est absent ou si aucune médiane ne peut être calculée pour
cette catégorie, nous utilisons la médiane globale de l’indicateur dans
l’entraînement.

Les valeurs imputées sont des approximations. Cette méthode suppose que
la médiane du groupe fournit un remplacement raisonnable et ne permet
pas de connaître la valeur réelle qui aurait été observée.

In [7]:
# Apprendre les médianes de chaque indicateur sur l'entraînement uniquement.
medianes_par_statut = X_train.groupby("status")[variables_numeriques].median()
medianes_globales = X_train[variables_numeriques].median()

def imputer_par_statut(X):
    """Imputer les valeurs numériques avec les médianes de l'entraînement."""
    X_impute = X.copy()

    for colonne in variables_numeriques:
        # Chercher la médiane correspondant au statut de chaque ligne.
        medianes_du_statut = X_impute["status"].map(medianes_par_statut[colonne])

        # Utiliser la médiane globale si la médiane du statut est indisponible.
        X_impute[colonne] = (
            X_impute[colonne]
            .fillna(medianes_du_statut)
            .fillna(medianes_globales[colonne])
        )

    return X_impute

# Réutiliser les mêmes médianes pour les deux ensembles.
X_train_impute = imputer_par_statut(X_train)
X_val_impute = imputer_par_statut(X_val)

# Examiner les médianes par statut et vérifier le résultat.
display(medianes_par_statut.T.round(2))

for nom, X_impute in [
    ("Entraînement", X_train_impute),
    ("Validation", X_val_impute),
]:
    nb_manquantes = X_impute[variables_numeriques].isna().sum().sum()
    print(f"{nom} : {nb_manquantes} valeurs numériques encore manquantes")

status,Developed,Developing
adult_mortality,75.50,166.00
alcohol,10.52,2.85
hepatitis_b,95.00,91.00
polio,96.00,89.00
diphtheria,96.00,90.00
hiv_aids,0.10,0.10
gdp,28279.57,1947.01
population,8026929.50,7666129.00
thinness_10_19_years,1.10,4.50
thinness_5_9_years,1.00,4.50


Entraînement : 0 valeurs numériques encore manquantes
Validation : 0 valeurs numériques encore manquantes


#### Résultat de l’imputation par statut

Les médianes calculées sur l’entraînement diffèrent selon le statut de
développement, notamment pour le PIB et les dépenses de santé par habitant.
L’imputation applique donc une valeur de référence propre à chaque catégorie
lorsqu’elle est disponible.

Après application des médianes apprises sur 2000–2011, aucune valeur
numérique ne reste manquante dans l’entraînement ni dans la validation.
Ce contrôle confirme que le traitement remplit les valeurs manquantes.
Sa pertinence pour la prédiction sera évaluée à partir des erreurs du
modèle sur la validation.

### Créer le module depuis le notebook LEP-20
Cette cellule utilise la méthode avec open() et write() pour enregistrer la fonction d’imputation dans le fichier pretraitement_lep.py. En exécutant la cellule depuis le dossier notebooks/, le module est créé à côté du notebook LEP-20. Les notebooks suivants pourront importer la même fonction et l’appliquer à leurs données en utilisant les médianes calculées sur leur ensemble d’entraînement.

In [8]:
code = '''"""Prétraitement des données du projet WHO Life Expectancy."""


def imputer_par_statut(X_reference, X_a_imputer, variables_numeriques):
    """Remplace les valeurs numériques manquantes par des médianes.

    Pour chaque indicateur, la fonction utilise la médiane du statut de
    développement. Si cette médiane est indisponible, elle utilise la
    médiane globale. Les deux sont calculées sur X_reference.

    Args:
        X_reference: Données d’entraînement utilisées pour calculer les médianes.
        X_a_imputer: Données à traiter, contenant la colonne `status`.
        variables_numeriques: Noms des colonnes numériques à imputer.

    Returns:
        Une copie de X_a_imputer avec les valeurs numériques imputées.
    """
    medianes_par_statut = (
        X_reference.groupby("status")[variables_numeriques].median()
    )
    medianes_globales = X_reference[variables_numeriques].median()

    X_impute = X_a_imputer.copy()

    for colonne in variables_numeriques:
        medianes_du_statut = X_impute["status"].map(medianes_par_statut[colonne])
        X_impute[colonne] = (
            X_impute[colonne]
            .fillna(medianes_du_statut)
            .fillna(medianes_globales[colonne])
        )

    return X_impute
'''

with open("pretraitement_lep.py", "w", encoding="utf-8") as fichier:
    fichier.write(code)